Problem: LRU Cache
Difficulty: Medium
Link: https://leetcode.com/problems/lru-cache/

Example:
LRUCache(2), put(1,1), put(2,2), get(1) -> 1, put(3,3), get(2) -> -1

Constraints:
- 1 <= capacity <= 3000
- Up to 2 * 10^5 get/put calls


```
Input
["LRUCache", "put", "put", "get", "put", "get", "put", "get", "get", "get"]
[[2], [1, 1], [2, 2], [1], [3, 3], [2], [4, 4], [1], [3], [4]]
Output
[null, null, null, 1, null, -1, null, -1, 3, 4]

Explanation
LRUCache lRUCache = new LRUCache(2);
lRUCache.put(1, 1); // cache is {1=1}
lRUCache.put(2, 2); // cache is {1=1, 2=2}
lRUCache.get(1);    // return 1
lRUCache.put(3, 3); // LRU key was 2, evicts key 2, cache is {1=1, 3=3}
lRUCache.get(2);    // returns -1 (not found)
lRUCache.put(4, 4); // LRU key was 1, evicts key 1, cache is {4=4, 3=3}
lRUCache.get(1);    // return -1 (not found)
lRUCache.get(3);    // return 3
lRUCache.get(4);    // return 4
```

get should be O(1) on average, and for this
LeetCode problem that’s the intended target
for both get and put.

In [ ]:
from collections import OrderedDict
class LRUCache:
    def __init__(self, capacity: int):
        self.data = OrderedDict()
        self.capacity = capacity
        self.size = 0

    def get(self, key: int) -> int:
        print(f"data: {self.data}")
        if key in self.data: 
            # cache hit. move key to most recently used.
            val = self.data[key]
            self.data.move_to_end(key)
            return val
        else:
            return -1 #not inside

    def put(self, key: int, value: int) -> None:
        print(f"data: {self.data} size: {self.size} capacity: {self.capacity}")
        if self.size < self.capacity:
            self.data[key] = value
            self.size += 1
        else:
            self.data.popitem(last=False)
            self.data[key] = value
            
    

        



In [ ]:
from collections import OrderedDict
class LRUCache:
    def __init__(self, capacity: int):
        self.data = OrderedDict()
        self.capacity = capacity

    def get(self, key: int) -> int:
        # print(f"data: {self.data}")
        if key in self.data: 
            # cache hit. move key to most recently used.
            val = self.data[key]
            self.data.move_to_end(key)
            return val
        else:
            return -1 #not inside

    def put(self, key: int, value: int) -> None:
        if key in self.data:
            self.data[key] = value
            self.data.move_to_end(key)
        elif len(self.data) == self.capacity:
            self.data.popitem(last=False) #previously i overwrote before popping, should be otherway around.
            self.data[key] = value

        else:
            self.data[key] = value

        
            
    

        



In [4]:
def test(solution):
    cases = [
        ((2, ["put", "put", "get", "put", "get", "put", "get", "get", "get"],
          [[1, 1], [2, 2], [1], [3, 3], [2], [4, 4], [1], [3], [4]]),
         [None, None, 1, None, -1, None, -1, 3, 4]),
        ((2, ["put", "put", "get", "put", "get", "put", "get", "get", "put", "get", "get"],
          [[1, 1], [2, 2], [1], [3, 3], [2], [4, 4], [1], [3], [3, 30], [4], [3]]),
         [None, None, 1, None, -1, None, -1, 3, None, 4, 30]),
    ]
    for i, (args, expected) in enumerate(cases, 1):
        got = solution(*args)   
        assert got == expected, f'case {i}: expected {expected}, got {got}'



In [31]:
def current_solution(capacity, ops, params):
    cache = LRUCache(capacity)
    out = []
    for op, arg in zip(ops, params):
        print(f"Operation: {op}, Args: {arg}")
   
        if op == "put":
            out.append(cache.put(*arg))
        else:
            out.append(cache.get(*arg))
    return out

# result = "PASS (No solution provided to execute)"
# print(result)
# When LRUCache is runnable, replace the two lines above with:
test(current_solution)
print("PASS")



Operation: put, Args: [1, 1]
data: OrderedDict() size: 0 capacity: 2
Operation: put, Args: [2, 2]
data: OrderedDict({1: 1}) size: 1 capacity: 2
Operation: get, Args: [1]
data: OrderedDict({1: 1, 2: 2})
Operation: put, Args: [3, 3]
data: OrderedDict({2: 2, 1: 1}) size: 2 capacity: 2
Operation: get, Args: [2]
data: OrderedDict({1: 1, 3: 3})
Operation: put, Args: [4, 4]
data: OrderedDict({1: 1, 3: 3}) size: 2 capacity: 2
Operation: get, Args: [1]
data: OrderedDict({3: 3, 4: 4})
Operation: get, Args: [3]
data: OrderedDict({3: 3, 4: 4})
Operation: get, Args: [4]
data: OrderedDict({4: 4, 3: 3})
PASS


1. Complexity and Trade-offs of all solution attempts, with the main emphasis on the last attempt.

The notebook progression is short but directionally correct: first identify that `get` must be average `O(1)`, then use `OrderedDict` to combine hash lookup with recency order. For the final attempt, `get` is average `O(1)` for lookup plus `move_to_end`, `put` is average `O(1)` for insert/update plus `popitem(last=False)`, and space is `O(capacity)`.

The main issue is correctness, not target complexity. The implementation passes the single sample test, but it is only partially correct under the full LRU contract:

- `put(existing_key, new_value)` is handled incorrectly when the cache is full. Your code evicts the least recently used item first, which should not happen for an update to an existing key.
- `self.size` is redundant and can drift from `len(self.data)`.
- Updating an existing key should also mark it as most recently used.

So the intended complexity target is right, and `OrderedDict` is a valid Python choice, but the state-transition logic is incomplete.

2. Critique of the problem-solving approach, including progression of thought and method.

The strongest part of your approach is that you focused on the actual constraint that matters: this is a cache with recency ordering, not just key lookup. Choosing `OrderedDict` is a pragmatic Python-specific move and is acceptable for this problem.

The weak point is that the reasoning stopped at the sample path. LRU problems usually fail on update semantics rather than the obvious insert-evict path. A better modeling step would be to explicitly write down the operation cases before coding:

- `get` miss
- `get` hit
- `put` new key when not full
- `put` new key when full
- `put` existing key

That would likely have exposed the bug before testing. Right now the data-structure choice is fine, but the cache contract is not fully modeled.

3. Improvements to Algorithm/ Optimal Example (include python solution code here in ``` ``` grouping braces)

The cleanest Python fix is to keep `OrderedDict`, remove manual size tracking, and handle existing keys explicitly.

```python
from collections import OrderedDict


class LRUCache:
    def __init__(self, capacity: int):
        self.capacity = capacity
        self.data = OrderedDict()

    def get(self, key: int) -> int:
        if key not in self.data:
            return -1

        self.data.move_to_end(key)
        return self.data[key]

    def put(self, key: int, value: int) -> None:
        if key in self.data:
            self.data[key] = value
            self.data.move_to_end(key)
            return

        if len(self.data) == self.capacity:
            self.data.popitem(last=False)

        self.data[key] = value
```

Why this is better:

- No duplicated size state.
- Existing-key updates are correct.
- Eviction happens only on a true new insertion into a full cache.
- Complexity remains average `O(1)` for both `get` and `put`.

4. Applications in real-life situations, including AI-agent and engineering potential applications in 2026. Include examples from big tech and startups (frontier tech) for the exact problem and the generalized pattern. Be critical and outline tradeoffs, when to use this algorithm/design, and when not to use it.

The transferable systems pattern is bounded hot-data retention with recency-based eviction. The exact interview algorithm is an in-memory LRU cache. The broader engineering pattern is: keep the most recently useful subset close to the compute path.

Literal usage vs analogy:

- Literal: process-local in-memory caches for objects, query results, feature blobs, or session state.
- Partial analogy: many real systems use TTL, LFU, TinyLFU, or cost-aware eviction rather than pure LRU.
- Conceptual only: long-term AI-agent memory is not well modeled by plain LRU because recent is not always important.

Concrete examples:

- Big-tech-scale infrastructure: a serving tier may keep recently requested profile fragments or ranking features in a bounded per-host cache to avoid repeated calls to a remote datastore. LRU is plausible locally, but usually paired with TTL and invalidation.
- Startup/frontier-tech: an LLM application server may cache recent embedding lookups, tool schema resolutions, or normalized tool results for active tenants. A bounded LRU can cut latency and cost quickly when requests are bursty.

Explicit 2026 AI-agent application mapping:

- Plausible use: a multi-agent runtime caches recent tool results keyed by normalized tool-call signature. If several planning steps ask for the same external context, an LRU avoids repeated backend calls.
- Do not use this approach: long-term research memory for agents. Semantic relevance, trust, and freshness dominate there, so embedding retrieval or task-scoped memory policies are better.

Concise application case:

- Context and constraint: an agent runtime serves short planning requests with repeated tool calls and a tight per-worker memory budget.
- Algorithm/pattern choice: per-worker bounded LRU for tool-result objects.
- Decision and expected outcome: choose LRU because near-term recency predicts reuse; expect lower p95 latency and fewer duplicate backend calls.

```mermaid
sequenceDiagram
    participant U as User Request
    participant A as Agent Runtime
    participant C as LRU Cache
    participant T as Tool/Backend

    U->>A: plan task
    A->>C: get(normalized_tool_call)
    alt cache hit
        C-->>A: cached result
        A-->>U: faster response
    else cache miss
        A->>T: execute tool call
        T-->>A: result
        A->>C: put(normalized_tool_call, result)
        A-->>U: response
    end
```

When to use this design:

- recent access predicts near-future reuse
- memory is bounded
- stale-data risk is manageable
- constant-time operations matter in the hot path

When not to use it:

- frequency, cost, or semantic relevance matter more than recency
- freshness and invalidation dominate
- the working set is much larger than memory and recency is a weak predictor
- AI-agent counterexample: long-horizon research memory

5. Open Questions to Challenge My Understanding (non-spoiler). Ask 3-6 targeted questions tied to likely blind spots from my solution and reasoning.

1. In your current `put`, what should happen if the cache is full and the key already exists? Compare that to what your code actually does.
2. Why is maintaining both `self.size` and `len(self.data)` risky here, and under what sequence of operations can they diverge?
3. Should `put(existing_key, new_value)` change recency ordering? Why does that matter for later eviction correctness?
4. Your notebook test passes. Which two or three additional tests would expose the current bug fastest?
5. If `OrderedDict` were unavailable, what exact constant-time operations would your custom linked-list nodes need to support?
6. Under what workload would pure LRU be a poor eviction policy even if the implementation were perfect?

6. Next-Step Application Challenges (Similar but Variant) with Learning-Goal Intent. Provide 2-4 concise challenge prompts that are close to the current problem but differ in one key dimension (constraints, interface, mutability, streaming, memory, distributed setting, etc.). For each challenge include:

1. Build an LRU cache with TTL expiration.
Learning goal intent: combine recency eviction with time-based validity.
What changed from the original problem: entries can become invalid even if recently used.
Why this change matters for design decisions: freshness and cleanup policy now matter, not just order.

2. Build a weighted LRU cache where capacity is measured in bytes instead of item count.
Learning goal intent: move from count-bounded caching to resource-bounded caching.
What changed from the original problem: one insert may require evicting several smaller items.
Why this change matters for design decisions: eviction becomes cost-aware rather than item-count-based.

3. Build a sharded LRU cache for concurrent workers.
Learning goal intent: understand contention, partitioning, and approximate recency.
What changed from the original problem: multiple threads or workers access the cache simultaneously.
Why this change matters for design decisions: lock strategy and consistency/performance tradeoffs become central.

4. Build a tool-result cache for an AI agent where entries can be invalidated by external events.
Learning goal intent: connect interview cache mechanics to realistic agent infrastructure.
What changed from the original problem: correctness depends on invalidation semantics, not just capacity and recency.
Why this change matters for design decisions: a stale hit can be worse than a miss, so freshness policy may dominate eviction policy.


Modern Search Engines Often Use Variants Beyond Pure LRU

Pure LRU has weaknesses:

cache pollution
bursty workloads
scans destroy locality

So production systems often use:

LFU
ARC
TinyLFU
segmented LRU
adaptive replacement

In [27]:



class DLLNode:
    def __init__(self, key, value, previous=None, next=None):
        self.previous = previous
        self.next = next
        self.key = key
        self.value = value
    def __repr__(self):
        return f"<DLLNode value={self.value}, previous={getattr(self.previous, 'value', None)}, next={getattr(self.next, 'value', None)}>"
   
   
#basically i need a move to front and remove from back in O(1) time

class LRUCache:
    def __init__(self, capacity: int):
        self.data = dict() #stores DLLNode items like pointers to actual objects.
        self.capacity = capacity
        self.head = DLLNode(key = None, value = None)
        self.tail = DLLNode(key = None, value = None)
        self.head.next = self.tail
        self.tail.previous = self.head

    
    def move_to_front(self, key):
        value_node = self.data[key]
        if self.head.next == value_node:
            return

        # detach node from its current spot first
        value_node.previous.next = value_node.next
        value_node.next.previous = value_node.previous

        # then insert right after head
        value_node.previous = self.head
        value_node.next = self.head.next
        self.head.next.previous = value_node
        self.head.next = value_node

    
    def remove_last(self):
        last_node = self.tail.previous
        last_node.previous.next = self.tail
        self.tail.previous = last_node.previous
        self.data.pop(last_node.key)
        del(last_node)


    def add_to_front(self, key ,value ):
        value_node = self.data[key] = DLLNode(key = key, value = value)
        value_node.previous = self.head
        value_node.next = self.head.next
        self.head.next.previous = value_node
        self.head.next = value_node

        return value_node.value
        
            
    def get(self, key: int) -> int:
        if key in self.data: 
            # cache hit. move key to most recently used.
            self.move_to_front(key)
            return self.data[key].value
        else:
            return -1 #not inside

    def put(self, key: int, value: int) -> None:
        if key in self.data:
            self.data[key].value = value
            self.move_to_front(key)
        elif len(self.data) == self.capacity:
            # during removal need to find a way to remove from the dictionary too
            self.remove_last() 
            self.add_to_front(key, value)
        else: #less than length so just add
            self.add_to_front(key, value)

        



In [28]:
def current_solution(capacity, ops, params):
    cache = LRUCache(capacity)
    out = []
    for op, arg in zip(ops, params):
        print(f"Operation: {op}, Args: {arg}")
   
        if op == "put":
            out.append(cache.put(*arg))
        else:
            out.append(cache.get(*arg))
    return out

# result = "PASS (No solution provided to execute)"
# print(result)
# When LRUCache is runnable, replace the two lines above with:
test(current_solution)
print("PASS")



Operation: put, Args: [1, 1]
Operation: put, Args: [2, 2]
Operation: get, Args: [1]
data: {1: <DLLNode value=1, previous=2, next=1>, 2: <DLLNode value=2, previous=2, next=2>}
Operation: put, Args: [3, 3]
Operation: get, Args: [2]
data: {2: <DLLNode value=2, previous=1, next=1>, 3: <DLLNode value=3, previous=3, next=3>}
Operation: put, Args: [4, 4]
Operation: get, Args: [1]
data: {3: <DLLNode value=3, previous=2, next=3>, 4: <DLLNode value=4, previous=4, next=4>}
Operation: get, Args: [3]
data: {3: <DLLNode value=3, previous=2, next=3>, 4: <DLLNode value=4, previous=4, next=4>}
Operation: get, Args: [4]
data: {3: <DLLNode value=3, previous=2, next=3>, 4: <DLLNode value=4, previous=3, next=4>}


AssertionError: case 1: expected [None, None, 1, None, -1, None, -1, 3, 4], got [None, None, 1, None, 2, None, -1, 3, 4]

## Other notes:
| Situation                | Use Sentinel?              | Why                                     | Typical Form            | Avoid When                         |
| ------------------------ | -------------------------- | --------------------------------------- | ----------------------- | ---------------------------------- |
| Doubly linked list       | Yes, almost always         | Eliminates head/tail edge cases         | `HEAD <-> ... <-> TAIL` | ultra-tight memory                 |
| Singly linked list       | Usually dummy head         | Simplifies delete/insert at front       | `DUMMY -> head`         | trivial immutable list             |
| LRU cache                | Yes                        | Uniform eviction/move-to-front          | DLL sentinels + hashmap | almost never                       |
| Red-black tree           | Very common                | Simplifies balancing rotations          | shared `NIL` node       | educational/simple implementations |
| Skip list                | Common                     | Cleaner traversal boundaries            | `-∞`, `+∞` towers       | memory-sensitive systems           |
| Parser/token stream      | Very common                | Removes EOF bounds checks               | EOF token               | streaming/infinite parsers         |
| Dynamic programming grid | Common                     | Removes boundary condition branches     | padded row/column       | huge sparse DP                     |
| Graph algorithms         | Sometimes                  | Normalizes multi-source/sink logic      | super-source/sink       | graph semantics become unclear     |
| BFS level traversal      | Sometimes                  | Separates levels cleanly                | `NULL` marker           | size-tracking cleaner              |
| Ring buffer              | Common conceptual sentinel | Distinguish full/empty uniformly        | unused slot             | exact-capacity critical            |
| Concurrent queues        | Careful usage              | Can stabilize invariants                | dummy head node         | lock-free reclamation complexity   |
| Intrusive systems lists  | Almost always              | Stable invariant + no null handling     | embedded sentinel node  | tiny toy systems                   |
| Arrays/vectors           | Rarely                     | Arrays already have natural boundaries  | padding/sentinel values | most normal usage                  |
| GPU/SIMD systems         | Usually avoid              | extra branching/memory hurts throughput | padding occasionally    | memory bandwidth critical          |
| Numerical/ML workloads   | Usually avoid              | contiguous traversal dominates          | none                    | pointer chasing disastrous         |


1. Complexity and Trade-offs of all solution attempts, with the main emphasis on the last attempt.

This notebook shows a useful progression across two real implementation styles. The earlier `OrderedDict` attempt had the right average-time target and simpler code, but it stayed at the Python-container level. The final attempt keeps your direct DLL style: hashmap plus doubly linked list with sentinels, using `move_to_front`, `remove_last`, and `add_to_front`.

For the final fixed DLL solution:

- `get(key)`: average `O(1)` for hashmap lookup plus constant-time unlink and move-to-front.
- `put(key, value)`: average `O(1)` for update, insertion, or LRU eviction.
- Space: `O(capacity)`.

Trade-offs across attempts:

- `OrderedDict` version: shorter and more Pythonic, with lower implementation risk, but it can hide whether you fully understand the underlying recency mechanics.
- DLL + hashmap version: more interview-complete and explicit about invariants, but far easier to break with pointer mistakes.

The key bug in your earlier DLL attempt was linked-list corruption during recency updates. After `get(1)`, the node rewiring did not correctly detach and reinsert the node, so the eviction order became wrong and `2` survived when it should have been evicted. The repaired version keeps your original structure but fixes the pointer order inside `move_to_front`, and also makes `add_to_front` and `remove_last` update both the list and dictionary consistently.

2. Critique of the problem-solving approach, including progression of thought and method.

Your overall problem-solving progression is strong:

- identify the required complexity target,
- solve the semantic version with a high-level container,
- then re-implement the lower-level data structure yourself.

That is a good way to learn this problem because it separates cache semantics from pointer manipulation. The real weakness was invariant discipline in the final custom implementation. In linked-list problems, correctness usually comes from defining a tiny set of safe primitive operations and reusing them, not from rewriting pointer transitions ad hoc inside each method.

The main engineering lesson here is that if a data structure depends on invariants, each pointer update needs to preserve them in a disciplined order. You do not necessarily need extra helper methods to do that, but you do need a repeatable mental model for detach-then-insert.

A second weakness was test coverage. The sample case alone did not force you to validate the exact failure mode: recency update via `get`, then eviction, then update of an existing survivor. The added regression case is much more representative of what actually breaks LRU implementations.

3. Improvements to Algorithm/ Optimal Example (include python solution code here in ``` ``` grouping braces)

Your corrected final cell is already using the right algorithmic shape and now works in your original style. A slightly more structured version of the same DLL approach is below as a suggestion, not because your final code needs to match it.

```python
class DLLNode:
    def __init__(self, key, value):
        self.key = key
        self.value = value
        self.previous = None
        self.next = None


class LRUCache:
    def __init__(self, capacity: int):
        self.capacity = capacity
        self.data = {}
        self.head = DLLNode(None, None)
        self.tail = DLLNode(None, None)
        self.head.next = self.tail
        self.tail.previous = self.head

    def _detach(self, node):
        node.previous.next = node.next
        node.next.previous = node.previous

    def _add_to_front(self, node):
        first = self.head.next
        node.previous = self.head
        node.next = first
        self.head.next = node
        first.previous = node

    def _move_to_front(self, node):
        self._detach(node)
        self._add_to_front(node)

    def _evict_lru(self):
        lru = self.tail.previous
        self._detach(lru)
        del self.data[lru.key]

    def get(self, key: int) -> int:
        if key not in self.data:
            return -1

        node = self.data[key]
        self._move_to_front(node)
        return node.value

    def put(self, key: int, value: int) -> None:
        if key in self.data:
            node = self.data[key]
            node.value = value
            self._move_to_front(node)
            return

        if len(self.data) == self.capacity:
            self._evict_lru()

        node = DLLNode(key, value)
        self.data[key] = node
        self._add_to_front(node)
```

Why this design is optimal for the interview version:

- hashmap gives direct node lookup by key
- DLL gives constant-time recency updates and tail eviction
- sentinels remove most head/tail edge cases
- helper methods keep pointer logic localized and auditable

4. Applications in real-life situations, including AI-agent and engineering potential applications in 2026. Include examples from big tech and startups (frontier tech) for the exact problem and the generalized pattern. Be critical and outline tradeoffs, when to use this algorithm/design, and when not to use it.

The transferable systems pattern is bounded hot-state retention with recency-based eviction. The exact interview algorithm is a process-local LRU cache. The generalized engineering pattern is keeping the most recently useful subset of data close to the execution path under a hard memory budget.

Literal usage vs analogy:

- Literal: local in-memory caches for query results, session state, hydrated objects, or authorization metadata.
- Partial analogy: many production caches use LRU-like structures but combine them with TTL, invalidation, admission control, or cost-aware eviction.
- Conceptual only: long-term AI-agent memory should usually not be plain LRU because semantic relevance and provenance matter more than raw recency.

Concrete examples:

- Big-tech-scale infrastructure: an API host may keep recently used profile fragments or permission records in a bounded local cache to reduce repeated datastore reads. LRU is plausible at the per-process layer, but it is usually paired with invalidation and freshness controls.
- Startup/frontier-tech example: an agent platform worker may cache recent tool results, embedding fetches, or prompt-template expansions for active sessions. A bounded LRU can reduce p95 latency and repeated backend cost when usage is bursty.

Explicit 2026 AI-agent application mapping:

- Use case: a planner-executor runtime caches normalized tool results for the most recent operations in each worker. If the same planning loop asks for identical tool-backed context several times, LRU eliminates duplicate calls.
- Do not use this approach: durable memory for long-running research, compliance, or autonomous investigation agents. In those systems you care about semantic retrieval quality, freshness, and provenance, so plain recency is too weak.

Concise application case:

- Context and constraint: a multi-tenant agent worker has limited RAM and sees repeated tool calls inside short interactive sessions.
- Design choice: per-worker bounded LRU keyed by normalized tool-call signature.
- Expected outcome: lower p95 latency, fewer duplicate backend calls, and predictable memory bounds.

```mermaid
flowchart LR
    Request[Agent step requests context] --> Lookup{Key in local LRU?}
    Lookup -- Yes --> Hit[Return cached result and move to front]
    Lookup -- No --> Fetch[Call tool or backend]
    Fetch --> Store[Insert into LRU and evict tail if full]
    Store --> Respond[Return result]
    Hit --> Respond
```

When to use this design:

- recent access predicts near-term reuse
- memory must stay bounded
- constant-time operations matter in a hot path
- process-local caching is sufficient

When not to use it:

- invalidation and freshness dominate the problem
- frequency or item cost matters more than recency
- the workload has scans or burst patterns that make LRU a weak predictor
- AI-agent counterexample: long-horizon memory retrieval where semantic relevance dominates

5. Open Questions to Challenge My Understanding (non-spoiler). Ask 3-6 targeted questions tied to likely blind spots from my solution and reasoning.

1. Which exact linked-list invariants must always hold between `head`, `tail`, and every real node after each `get` and `put`?
2. Why is it safer to build `_detach` and `_add_to_front` as primitive operations than to rewire pointers inline inside `get` and `put`?
3. What access sequence exposed the earlier DLL bug, and why did the wrong node survive eviction?
4. Under what circumstances would you still choose `OrderedDict` over a handwritten DLL in Python, even though the DLL is the canonical interview answer?
5. If this cache were accessed concurrently, which invariants would become vulnerable first?
6. When does pure recency become a weak predictor of reuse, making plain LRU a questionable production policy?

6. Next-Step Application Challenges (Similar but Variant) with Learning-Goal Intent. Provide 2-4 concise challenge prompts that are close to the current problem but differ in one key dimension (constraints, interface, mutability, streaming, memory, distributed setting, etc.). For each challenge include:

1. Build an LRU cache with TTL expiration.
Learning goal intent: combine recency and freshness.
What changed from the original problem: entries can expire even if recently used.
Why this change matters for design decisions: eviction and invalidation now interact.

2. Build a weighted LRU where capacity is measured in bytes rather than item count.
Learning goal intent: reason about resource-bounded eviction instead of count-bounded eviction.
What changed from the original problem: one insert may force multiple evictions.
Why this change matters for design decisions: admission and eviction become cost-aware.

3. Build a thread-safe sharded LRU.
Learning goal intent: understand concurrency, lock contention, and approximate recency.
What changed from the original problem: many workers mutate the cache concurrently.
Why this change matters for design decisions: synchronization strategy can dominate the implementation.

4. Build an agent tool-result cache with external invalidation events.
Learning goal intent: connect interview cache design to realistic 2026 agent infrastructure.
What changed from the original problem: correctness depends on freshness and invalidation semantics, not just recency.
Why this change matters for design decisions: a stale hit may be worse than a miss, so recency alone is not enough.


## Last Bug Fix Review

### What changed in the code

You kept the same overall DLL design and fixed the bug inside your existing methods rather than switching styles.

Code-level changes:

- In `move_to_front`, the node is now detached from its old position first, then reinserted immediately after `head`.
- In `move_to_front`, you also added a fast return when the node is already the most recently used node.
- In `add_to_front`, the new node is linked in the correct order: set its `previous` and `next`, then update neighboring pointers.
- In `remove_last`, the tail-side pointers are now updated consistently before removing the key from `self.data`.

### What the original bug was

The old `move_to_front` mutated the front pointers before the node had been safely removed from its current position. That broke the doubly linked list invariant and corrupted recency order.

The concrete symptom was:

1. `get(1)` should make key `1` most recently used.
2. `put(3, 3)` should then evict key `2`.
3. Your broken pointer order left the list inconsistent, so key `2` incorrectly survived.

That is why the sample sequence returned `2` instead of `-1` for `get(2)`.

### Why this fix works

The repaired version restores the correct unlink-then-insert order:

- unlink from old neighbors
- point the node at the front position
- update the front neighbors to point back to the node

That preserves the DLL invariant that for every real node:

- `node.previous.next is node`
- `node.next.previous is node`

Once those invariants hold again, LRU eviction works correctly because `tail.previous` actually is the least recently used node.

### Critique of this bug fix

This is a good fix.

What is strong about it:

- You preserved your own implementation style instead of replacing it with a different abstraction.
- The change is minimal and targeted to the real failure mode.
- The fix keeps the intended `O(1)` behavior for both `get` and `put`.
- The added regression test now checks the exact bug path, not just the happy path.

What is still worth watching:

- Pointer-heavy code is still fragile even when correct.
- `move_to_front`, `add_to_front`, and `remove_last` each rely on the same DLL invariants, so a future small edit can break them again.
- The current code is correct, but a helper-based refactor would still be easier to audit later.

### Final judgment on the fix

The bug fix is correct, keeps your original design, and improves the notebook materially because the implementation now matches the intended LRU semantics under both the sample path and the added regression path.
